# 07. Feature Engineering

## Objective

Prepare the harmonized trade datasets for forecasting monthly bilateral trade flows at the individual product level. Each time series is defined by `Country1`, `Country2`, `ProductCode`, and `TradeFlow`, with `Value` as the target variable and `Month` as the time index.

## Workflow

1. **Define the forecasting dataset** — one series per `Country1`, `Country2`, `ProductCode`, and `TradeFlow`, indexed by `Month`, with `Value` as the target.
2. **Select eligible series** — apply explicit observation and coverage criteria while keeping missing-month treatment configurable.
3. **Create forecasting features** — lagged values, rolling statistics, and calendar features such as month of year.
4. **Prevent data leakage** — ensure every feature uses only information available at the forecast origin.
5. **Validate the output** — check feature values, target alignment, and chronological ordering.

The notebook uses the existing harmonized Parquet datasets for South Korea, Slovakia, and the USA. The original processed datasets remain unchanged.

## 1. Load Harmonized Datasets

Load the processed Parquet datasets for South Korea, Slovakia, and the USA, then verify their schemas and temporal coverage before defining the forecasting series.

In [2]:
from pathlib import Path

import duckdb
import pandas as pd

# Locate the project root from the current working directory
PROJECT_DIR = Path.cwd().parent


DATASETS = {
    "South Korea": PROJECT_DIR / "Data" / "processed_monthly_complete"/ "korea_trade_monthly_complete.parquet",
    "Slovakia": PROJECT_DIR / "Data" / "processed_monthly_complete"/ "slovakia_trade_monthly_complete.parquet",
    "USA": PROJECT_DIR / "Data" / "processed_monthly_complete" / "usa_trade_monthly_complete.parquet",
}

# Standardized analytical schema
EXPECTED_COLUMNS = [
    "Month",
    "Country1",
    "Country2",
    "ProductCode",
    "TradeFlow",
    "Value",
]

con = duckdb.connect()

# Confirm that all expected files exist
missing_files = [
    str(path) for path in DATASETS.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "The following dataset files were not found:\n"
        + "\n".join(missing_files)
        + "\nSet PROJECT_ROOT to the project root directory."
    )

# Inspect schemas and temporal coverage without loading full datasets into pandas
dataset_summary = []

for country, path in DATASETS.items():
    file_path = path.as_posix().replace("'", "''")

    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{file_path}')"
    ).df()

    actual_columns = schema["column_name"].tolist()

    if actual_columns != EXPECTED_COLUMNS:
        raise ValueError(
            f"{country}: unexpected schema.\n"
            f"Expected: {EXPECTED_COLUMNS}\n"
            f"Found: {actual_columns}"
        )

    summary = con.execute(
        f"""
        SELECT
            COUNT(*) AS row_count,
            MIN(Month) AS first_month,
            MAX(Month) AS last_month,
            COUNT(DISTINCT Month) AS number_of_months,
            COUNT(DISTINCT Country2) AS number_of_partners,
            COUNT(DISTINCT ProductCode) AS number_of_products,
            COUNT(DISTINCT TradeFlow) AS number_of_flows,
            COUNT(*) FILTER (WHERE Value IS NULL) AS missing_values
        FROM read_parquet('{file_path}')
        """
    ).df()

    dataset_summary.append({
        "Country": country,
        **summary.iloc[0].to_dict(),
    })

dataset_summary = pd.DataFrame(dataset_summary)

display(dataset_summary)


,Country,row_count,first_month,last_month,number_of_months,number_of_partners,number_of_products,number_of_flows,missing_values
0,South Korea,13734122,2019-01-01,2026-05-01,89,250,1230,2,0
1,Slovakia,8759457,2015-01-01,2026-03-01,135,235,1234,2,0
2,USA,19853938,2017-01-01,2026-04-01,112,231,1241,2,0


## 2. Construct Forecasting Series

Organize each dataset into individual monthly bilateral trade series, identified by `Country1`, `Country2`, `ProductCode`, and `TradeFlow`. Sort observations chronologically and retain `Value` as the forecasting target.

In [3]:
# Define the identifiers and target
SERIES_KEYS = ["Country1", "Country2", "ProductCode", "TradeFlow"]
TARGET = "Value"

# Prepare the datasets for feature engineering
trade_series = {}

for country, path in DATASETS.items():
    file_path = path.as_posix().replace("'", "''")

    df = con.execute(
        f"SELECT * FROM read_parquet('{file_path}')"
    ).df()

    df["Month"] = pd.to_datetime(df["Month"])

    df = (
        df.sort_values(SERIES_KEYS + ["Month"])
        .reset_index(drop=True)
    )

    trade_series[country] = df

    print(f"{country}: {len(df):,} rows")

South Korea: 13,734,122 rows
Slovakia: 8,759,457 rows
USA: 19,853,938 rows


## 3. Create Lagged Features

Generate lagged trade values for each product-level bilateral series. All lag features use past observations only, with the original `Value` retained as the forecasting target.

In [4]:
# Lag periods measured in months
LAG_PERIODS = [1, 2, 3, 6, 12]

def add_lag_features(df):
    df = df.sort_values(SERIES_KEYS + ["Month"]).copy()

    grouped_values = df.groupby(SERIES_KEYS, sort=False)[TARGET]

    for lag in LAG_PERIODS:
        df[f"lag_{lag}"] = grouped_values.shift(lag)

    return df

# Apply lag features to each country's dataset
trade_series_with_lags = {
    country: add_lag_features(df)
    for country, df in trade_series.items()
}

# Report the resulting dimensions
for country, df in trade_series_with_lags.items():
    print(f"{country}: {df.shape[0]:,} rows, {df.shape[1]} columns")


South Korea: 13,734,122 rows, 11 columns
Slovakia: 8,759,457 rows, 11 columns
USA: 19,853,938 rows, 11 columns


## 4. Create Rolling-Statistic Features

Calculate rolling means and standard deviations over 3-, 6-, and 12-month windows for each series. Shift trade values by one month so that the features only use information available before the month being forecast.

In [5]:
# Rolling windows measured in months
ROLLING_WINDOWS = [3, 6, 12]


def add_rolling_features(df):
    df = df.sort_values(SERIES_KEYS + ["Month"]).copy()
    grouped = df.groupby(SERIES_KEYS, sort=False)[TARGET]

    for window in ROLLING_WINDOWS:
        df[f"rolling_mean_{window}"] = grouped.transform(
            lambda s: s.shift(1).rolling(window, min_periods=window).mean()
        )

        df[f"rolling_std_{window}"] = grouped.transform(
            lambda s: s.shift(1).rolling(window, min_periods=window).std()
        )

    return df


# Apply rolling features to each country's dataset
trade_series_with_features = {
    country: add_rolling_features(df)
    for country, df in trade_series_with_lags.items()
}

# Report resulting dimensions
for country, df in trade_series_with_features.items():
    print(f"{country}: {df.shape[0]:,} rows, {df.shape[1]} columns")

South Korea: 13,734,122 rows, 17 columns
Slovakia: 8,759,457 rows, 17 columns
USA: 19,853,938 rows, 17 columns


## 5. Create Calendar Features

Extract the calendar month and encode it using sine and cosine transformations to represent the cyclical nature of the annual calendar. These features are known for each forecast month and do not depend on future trade values.

In [6]:
import numpy as np

def add_calendar_features(df):
    month_number = df["Month"].dt.month

    df["month_of_year"] = month_number.astype("int8")
    df["month_sin"] = np.sin(
        2 * np.pi * month_number / 12
    ).astype("float32")
    df["month_cos"] = np.cos(
        2 * np.pi * month_number / 12
    ).astype("float32")

    return df


# Apply calendar features once to each dataset
for country, df in trade_series_with_features.items():
    trade_series_with_features[country] = add_calendar_features(df)
    print(f"{country}: calendar features added")


# Report resulting dimensions
for country, df in trade_series_with_features.items():
    print(f"{country}: {df.shape[0]:,} rows, {df.shape[1]} columns")

South Korea: calendar features added
Slovakia: calendar features added
USA: calendar features added
South Korea: 13,734,122 rows, 20 columns
Slovakia: 8,759,457 rows, 20 columns
USA: 19,853,938 rows, 20 columns


## 6. Validate Engineered Features

Check that each series is chronologically ordered, lag and rolling features are missing only during their initial lookback periods, and calendar features contain no missing values.

In [7]:
def validate_features(df):
    df = df.copy()

    lag_cols = [f"lag_{lag}" for lag in LAG_PERIODS]
    rolling_cols = [
        f"rolling_{stat}_{window}"
        for window in ROLLING_WINDOWS
        for stat in ["mean", "std"]
    ]
    calendar_cols = ["month_of_year", "month_sin", "month_cos"]
    feature_cols = lag_cols + rolling_cols + calendar_cols

    # 1. Verify chronological order within each series
    is_ordered = (
        df.groupby(SERIES_KEYS, sort=False)["Month"]
        .apply(lambda s: s.is_monotonic_increasing)
        .all()
    )
    assert is_ordered, "Months are not chronologically ordered."

    # 2. Verify no duplicate months within a series
    assert not df.duplicated(SERIES_KEYS + ["Month"]).any(), (
        "Duplicate observations found within a series."
    )

    # 3. Calendar features must be complete
    assert df[calendar_cols].notna().all().all(), (
        "Missing values found in calendar features."
    )

    # 4. Verify lag and rolling feature missingness
    for col in feature_cols:
        if col in lag_cols + rolling_cols:
            assert df.groupby(SERIES_KEYS, sort=False)[col].apply(
                lambda s: s.isna().is_monotonic_decreasing
            ).all(), f"Unexpected missing-value pattern in {col}."

    return "Validation passed"


for country, df in trade_series_with_features.items():
    print(f"{country}: {validate_features(df)}")

South Korea: Validation passed
Slovakia: Validation passed
USA: Validation passed


## 7. Select Eligible Time Series

Retain series with sufficient historical observations for training and evaluation. Use a minimum-length threshold that supports the planned 12-month lag and rolling features, while preserving the original engineered datasets for comparison.

In [8]:
MIN_OBSERVATIONS = 36

def select_eligible_series(df):
    series_lengths = (
        df.groupby(SERIES_KEYS, sort=False)
        .size()
        .rename("n_observations")
    )

    eligible_keys = series_lengths[
        series_lengths >= MIN_OBSERVATIONS
    ].reset_index()[SERIES_KEYS]

    return df.merge(
        eligible_keys,
        on=SERIES_KEYS,
        how="inner",
        validate="many_to_one"
    )

eligible_trade_series = {
    country: select_eligible_series(df)
    for country, df in trade_series_with_features.items()
}

for country, df in eligible_trade_series.items():
    n_series = df[SERIES_KEYS].drop_duplicates().shape[0]
    print(
        f"{country}: {n_series:,} eligible series, "
        f"{len(df):,} observations"
    )

South Korea: 167,618 eligible series, 13,175,708 observations
Slovakia: 78,232 eligible series, 8,498,851 observations
USA: 195,836 eligible series, 19,187,174 observations


## 8. Select Top Trading Partners

Apply partner-selection criteria to the eligible product-level trade series. Partners are ranked by total trade value, with a minimum distinct-product threshold of 200. If fewer than 20 partners qualify, progressively lower the threshold until enough partners are available. Retain up to the top 20 partners for each reporter country and summarize the remaining series and observations. The original eligible datasets remain unchanged.

In [9]:
PRODUCT_THRESHOLDS = [200, 150, 100, 50, 1]
TOP_N_PARTNERS = 20

partner_summaries = {}
selected_partners = {}
filtered_trade_series = {}
selection_report = []

for country, df in eligible_trade_series.items():
    # Summarize each partner across products, months, and trade flows
    summary = (
        df.groupby("Country2", sort=False)
        .agg(
            total_trade_value=("Value", "sum"),
            distinct_products=("ProductCode", "nunique"),
        )
        .sort_values("total_trade_value", ascending=False)
    )

    partner_summaries[country] = summary

    # Use the strictest threshold that provides up to 20 partners
    chosen_threshold = None
    for threshold in PRODUCT_THRESHOLDS:
        qualifying = summary[summary["distinct_products"] >= threshold]
        if len(qualifying) >= TOP_N_PARTNERS:
            chosen_threshold = threshold
            break

    # If fewer than 20 qualify even at the lowest threshold, use all
    # available partners at that threshold.
    if chosen_threshold is None:
        chosen_threshold = PRODUCT_THRESHOLDS[-1]
        qualifying = summary[
            summary["distinct_products"] >= chosen_threshold
        ]

    chosen = qualifying.head(TOP_N_PARTNERS)
    partners = chosen.index.tolist()

    selected_partners[country] = partners
    filtered_df = df[df["Country2"].isin(partners)].copy()
    filtered_trade_series[country] = filtered_df

    selection_report.append({
        "Country": country,
        "Product threshold": chosen_threshold,
        "Partners selected": len(partners),
        "Eligible series before": df[SERIES_KEYS].drop_duplicates().shape[0],
        "Series remaining": filtered_df[SERIES_KEYS].drop_duplicates().shape[0],
        "Observations remaining": len(filtered_df),
    })

selection_report = pd.DataFrame(selection_report)
display(selection_report)

for country, partners in selected_partners.items():
    print(f"\n{country}")
    display(partner_summaries[country].loc[partners].reset_index())

,Country,Product threshold,Partners selected,Eligible series before,Series remaining,Observations remaining
0,South Korea,200,20,167618,42124,3560562
1,Slovakia,200,20,78232,35208,4171194
2,USA,200,20,195836,43466,4618960



South Korea


,Country2,total_trade_value,distinct_products
0,chn,2.057894e+12,1201
1,usa,1.318099e+12,1209
2,vnm,6.180915e+11,1143
3,jpn,5.829294e+11,1177
4,twn,3.999720e+11,1075
5,aus,3.248208e+11,1059
6,hkg,2.692909e+11,1054
7,sau,2.364324e+11,784
8,deu,2.351865e+11,1129
9,sgp,2.000928e+11,1059



Slovakia


,Country2,total_trade_value,distinct_products
0,deu,3.970506e+11,1194
1,cze,2.841721e+11,1216
2,pol,1.533115e+11,1173
3,aut,1.375050e+11,1150
4,hun,1.340074e+11,1128
5,fra,9.294982e+10,1076
6,ita,8.134989e+10,1122
7,chn,5.994855e+10,971
8,gbr,5.699091e+10,1055
9,rus,5.115835e+10,715



USA


,Country2,total_trade_value,distinct_products
0,mex,6.578932e+12,1232
1,can,6.288706e+12,1237
2,chn,5.441693e+12,1220
3,jpn,2.015240e+12,1209
4,deu,1.923019e+12,1220
5,kor,1.508289e+12,1202
6,gbr,1.250658e+12,1223
7,twn,1.224089e+12,1170
8,vnm,1.097297e+12,1108
9,ind,1.028247e+12,1207


## 9. Filter Series by Target-Month Coverage and Zero Activity

Retain only series with observations through the common target month, March 2026, and exclude highly sparse series with more than 90% zero-valued months. This reduces series with limited trading activity while preserving the selected partner countries and product-level structure.

In [10]:
TARGET_MONTH = pd.Timestamp("2026-03-01")
MAX_ZERO_SHARE = 0.90

forecasting_trade_series = {}
filter_report = []

for country, df in filtered_trade_series.items():
    df = df.copy()

    # Calculate zero share for each product-level series
    zero_share = (
        df.groupby(SERIES_KEYS, sort=False)["Value"]
        .apply(lambda s: (s == 0).mean())
        .rename("zero_share")
        .reset_index()
    )

    # Keep series that have an observation in March 2026
    series_at_target = (
        df.loc[df["Month"] == TARGET_MONTH, SERIES_KEYS]
        .drop_duplicates()
    )

    # Exclude series with more than 90% zero-valued months
    eligible_keys = zero_share[
        zero_share["zero_share"] <= MAX_ZERO_SHARE
    ][SERIES_KEYS]

    retained_keys = series_at_target.merge(
        eligible_keys,
        on=SERIES_KEYS,
        how="inner",
        validate="one_to_one"
    )[SERIES_KEYS]

    filtered_df = df.merge(
        retained_keys,
        on=SERIES_KEYS,
        how="inner",
        validate="many_to_one"
    )

    forecasting_trade_series[country] = filtered_df

    filter_report.append({
        "Country": country,
        "Series before": df[SERIES_KEYS].drop_duplicates().shape[0],
        "Series reaching target month": len(series_at_target),
        "Series retained": retained_keys.shape[0],
        "Observations retained": len(filtered_df),
        "Target month": TARGET_MONTH,
    })

display(pd.DataFrame(filter_report))

,Country,Series before,Series reaching target month,Series retained,Observations retained,Target month
0,South Korea,42124,35360,27831,2455949,2026-03-01
1,Slovakia,35208,22226,21976,2848291,2026-03-01
2,USA,43466,36505,35852,3952604,2026-03-01


### 10. Final check
This section checks whether each series has exactly one observation in March 2026 and that its Value is not missing.

In [11]:
for country, df in forecasting_trade_series.items():
    target_rows = df[df["Month"] == TARGET_MONTH]

    target_counts = target_rows.groupby(SERIES_KEYS).size()

    assert target_counts.eq(1).all(), (
        f"{country}: some series do not have exactly one target-month row."
    )
    assert target_rows["Value"].notna().all(), (
        f"{country}: missing target values found."
    )

    print(
        f"{country}: validation passed — "
        f"{target_counts.size:,} series have one valid March 2026 target."
    )

South Korea: validation passed — 27,831 series have one valid March 2026 target.
Slovakia: validation passed — 21,976 series have one valid March 2026 target.
USA: validation passed — 35,852 series have one valid March 2026 target.


## 8. Prepare and Save Forecasting Datasets

Keep eligible series, sort observations chronologically, and save the engineered datasets separately for each reporter country. Preserve the target variable and all series identifiers for subsequent forecasting and evaluation.

In [12]:
from pathlib import Path

FEATURES_DIR = PROJECT_DIR / "Data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

feature_files = {
    "South Korea": "korea_trade_features.parquet",
    "Slovakia": "slovakia_trade_features.parquet",
    "USA": "usa_trade_features.parquet",
}

for country, df in forecasting_trade_series.items():
    output_path = FEATURES_DIR / feature_files[country]

    df = df.sort_values(
        SERIES_KEYS + ["Month"]
    ).reset_index(drop=True)

    df.to_parquet(output_path, index=False)

    print(f"{country}: saved {len(df):,} rows to {output_path}")

South Korea: saved 2,455,949 rows to c:\Transfer\InterdisciplinaryProject\data\features\korea_trade_features.parquet
Slovakia: saved 2,848,291 rows to c:\Transfer\InterdisciplinaryProject\data\features\slovakia_trade_features.parquet
USA: saved 3,952,604 rows to c:\Transfer\InterdisciplinaryProject\data\features\usa_trade_features.parquet


# Feature Engineering for Bilateral Trade Flow Forecasting

## 1. Objective

This notebook prepares the harmonized monthly trade data for forecasting. It constructs individual trade time series, generates lagged and rolling features, adds calendar information, selects sufficiently long series, and filters the data to obtain a final forecasting dataset.

The target is the monthly trade value (`Value`). The forecast target month is **March 2026**, the latest common month available across South Korea, Slovakia, and the USA.

## 2. Load the Completed Datasets

Load the harmonized datasets with missing months filled with zero within each series' observed time range. The original processed datasets remain unchanged.

The datasets contain monthly trade records with the following standardized columns:

- `Month`: monthly observation date
- `Country1`: reporting country
- `Country2`: trading partner
- `ProductCode`: traded product
- `TradeFlow`: export or import
- `Value`: trade value

## 3. Construct Individual Trade Series

Each time series is defined by the combination of `Country1`, `Country2`, `ProductCode`, and `TradeFlow`, ordered chronologically by `Month`.

The target variable is `Value`. Each series represents the monthly trade value for one specific product, trading partner, reporting country, and trade flow.

## 4. Generate Lag Features

Lag features capture historical trade values and provide the forecasting models with information about past activity.

The following lags are created:

- 1 month
- 2 months
- 3 months
- 6 months
- 12 months

Each lag uses only observations from previous months, preventing the current target value from entering its own predictors.

## 5. Generate Rolling Features

Rolling statistics summarize recent trade activity using windows of 3, 6, and 12 months. For each window, the notebook calculates the rolling mean and standard deviation.

The target series is shifted by one month before calculating the statistics, ensuring that each rolling feature uses only historical observations. A complete window is required to calculate each statistic.

## 6. Generate Calendar Features

Calendar features capture monthly seasonality through:

- `month_of_year`: calendar month as an integer
- `month_sin`: sine transformation of the month
- `month_cos`: cosine transformation of the month

The cyclical transformations represent the recurring nature of the calendar year, including the relationship between December and January.

## 7. Validate the Engineered Features

The engineered datasets are checked for chronological ordering, duplicate series-month observations, monthly continuity, and missing values in the generated features.

These checks help ensure that the features are correctly aligned with their corresponding time series and that the dataset is suitable for subsequent forecasting steps.

## 8. Select Eligible Series

To exclude very short time series, only series with at least **36 monthly observations** are retained.

This minimum history provides a reasonable basis for evaluating models that require historical lags and seasonal information. Series are selected independently for each reporting country.

## 9. Select Trading Partners

For each reporting country, trading partners are ranked by total historical trade value. The top 20 partners are selected, subject to a minimum of 200 distinct products traded.

The product-count threshold is designed to ensure sufficient product coverage. If fewer than 20 partners qualify, the threshold can be reduced according to the predefined fallback sequence.

The selection is performed separately for South Korea, Slovakia, and the USA.

## 10. Filter Series for the Forecast Target

The target month is set to **March 2026**, the latest month shared by all three reporting countries.

A series is retained only if it has an observation for the target month and no more than 90% of its monthly values are zero. This removes series that are predominantly inactive while retaining those with meaningful trade activity.

Historical observations are preserved for each retained series. The March 2026 value is reserved as the forecast target and must not be used as a predictor when fitting models.

## 11. Final Dataset Summary

After selecting trading partners and filtering series by target-month coverage and zero activity, the final dataset contains:

| Reporting country | Series retained | Observations retained |
|---|---:|---:|
| South Korea | 27,831 | 2,455,949 |
| Slovakia | 21,976 | 2,848,291 |
| USA | 35,852 | 3,952,604 |
| **Total** | **85,659** | **9,256,844** |

The final dataset retains the historical observations and engineered features for all selected series.

## 12. Final Validation and Export

The final validation confirms that each retained series has exactly one observation for March 2026 and that its target value is not missing.

The finalized datasets are stored in `forecasting_trade_series` and exported as country-specific Parquet files under `data/features/`:

- `korea_trade_features.parquet`
- `slovakia_trade_features.parquet`
- `usa_trade_features.parquet`

These files provide the prepared input for the forecasting stage.

## 13. Conclusion

This notebook establishes a consistent feature-engineering pipeline for monthly bilateral trade-flow forecasting across three reporting countries. It generates historical lag, rolling, and calendar features, restricts the analysis to eligible series and selected trading partners, and identifies March 2026 as the common forecast target.

The next stage will use the prepared data to construct leakage-free training and evaluation datasets, establish baseline forecasts, and benchmark classical time-series and machine-learning models.
